In [ ]:
!pip install pyspark py4j

In [57]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, mean, col, coalesce

spark = SparkSession.builder.appName("Read CSV Example").getOrCreate()

# Чтение CSV-файла
actors_df = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/actors.csv", header=True, inferSchema=True)
movie_actors_df = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/movie_actors.csv", header=True, inferSchema=True)
movies_df = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/movies.csv", header=True, inferSchema=True)


df = actors_df.join(movie_actors_df, on="actor_id", how="inner")
df = df.join(movies_df, on="movie_id", how="inner")

df.createOrReplaceTempView("movies_data")

top_genre_df = spark.sql("""
SELECT genre, count(distinct movie_id) as num_movies
FROM movies_data
group by genre
order by 2 desc
limit 5
""")

max_movies_df = spark.sql("""
SELECT name, count(distinct movie_id) as num_movies
FROM movies_data
group by name
order by 2 desc
limit 1
""")

avg_budget_df = spark.sql("""
SELECT genre, avg(budget) as avg_budget
FROM movies_data
group by genre
""")

country_movie_df = spark.sql("""
SELECT title, country, count(actor_id) as num_actors
FROM movies_data
group by title, country
having count(actor_id) > 1
""")

# Показ результатов
top_genre_df.show() # Найдите топ-5 жанров по количеству фильмов
max_movies_df.show() # Найдите актера с наибольшим количеством фильмов
avg_budget_df.show() # Подсчитайте средний бюджет фильмов по жанрам
country_movie_df.show() # Найдите фильмы, в которых снялось более одного актера из одной страны


+------+----------+
| genre|num_movies|
+------+----------+
|Action|         6|
| Drama|         4|
|Comedy|         4|
|Horror|         2|
|Sci-Fi|         2|
+------+----------+

+--------+----------+
|    name|num_movies|
+--------+----------+
|Actor_24|         5|
+--------+----------+

+------+--------------------+
| genre|          avg_budget|
+------+--------------------+
| Drama|6.2562771613076925E7|
|Horror|    8.711155335875E7|
|Comedy| 4.883974718111111E7|
|Action|2.5901128670000006E7|
|Sci-Fi| 7.929615028999999E7|
+------+--------------------+

+--------+---------+----------+
|   title|  country|num_actors|
+--------+---------+----------+
| Movie_7|    India|         2|
| Movie_3|      USA|         2|
|Movie_10|       UK|         2|
|Movie_15|    India|         2|
|Movie_18|Australia|         2|
| Movie_1|    India|         3|
| Movie_2|      USA|         2|
| Movie_7|      USA|         2|
|Movie_10|      USA|         2|
+--------+---------+----------+

